In [1]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib
import warnings

warnings.filterwarnings("ignore")

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]
  volume_df = df3["volume"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)
  volume_df = df3.xs("Volume", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")
volume_df = volume_df.dropna(axis =1 , how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns).intersection(volume_df.columns)
)
close_df, high_df, low_df, open_df, volume_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols],
    volume_df[valid_cols]
)




[**************        29%                       ]  143 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************65%******                 ]  326 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************72%**********             ]  360 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['HONA', 'FDXF', 'VYLR']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### True Strength Index crossover strategy

In [4]:
rsi_dict = {}
tsi_dict = {}
tsi_signal_dict = {}

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)


    rsi = talib.RSI(
            prices,
            timeperiod=14) 
    
    momentum = talib.MOM(prices, timeperiod=1)

    smoothed_momentum = talib.EMA(
        talib.EMA(momentum, timeperiod=25),
        timeperiod=13
    )

    absolute_momentum = np.abs(momentum)

    smoothed_absolute_momentum = talib.EMA(
        talib.EMA(absolute_momentum, timeperiod=25),
        timeperiod=13
    )

    tsi = 100 * smoothed_momentum / smoothed_absolute_momentum


    tsi = np.divide(
        100 * smoothed_momentum,
        smoothed_absolute_momentum,
        out=np.full_like(smoothed_momentum, np.nan),
        where=smoothed_absolute_momentum != 0
    )

    tsi_signal = talib.EMA(tsi, timeperiod=7)

    rsi_dict[ticker] = rsi
    tsi_dict[ticker] = tsi
    tsi_signal_dict[ticker] = tsi_signal


rsi_df = pd.DataFrame(rsi_dict, index=close_df.index)
tsi_df = pd.DataFrame(tsi_dict, index=close_df.index)
tsi_signal_df = pd.DataFrame(tsi_signal_dict, index=close_df.index)


entries = (
    (tsi_df > tsi_signal_df) &
    (tsi_df.shift(1) <= tsi_signal_df.shift(1))).shift(1).fillna(False).astype(bool)
exits = (
    (tsi_df < tsi_signal_df) &
    (tsi_df.shift(1) >= tsi_signal_df.shift(1))).shift(1).fillna(False).astype(bool)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()


Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               298227.965802
Total Return [%]                           198.227966
Benchmark Return [%]                       524.559266
Max Gross Exposure [%]                          100.0
Total Fees Paid                          75048.819318
Max Drawdown [%]                            26.960596
Max Drawdown Duration               400 days 00:00:00
Total Trades                                    56254
Total Closed Trades                             56019
Total Open Trades                                 235
Open Trade PnL                           16449.315421
Win Rate [%]                                41.482354
Best Trade [%]                             167.750682
Worst Trade [%]                            -43.553129
Avg Winning Trade [%]       

### TSI paired with RSI (and exit conditions)

In [5]:
entries = (
    (tsi_df > tsi_signal_df) &
    (tsi_df.shift(1) <= tsi_signal_df.shift(1)) & (rsi_df < 30)).shift(1).fillna(False).astype(bool)
exits = (
    (tsi_df < tsi_signal_df) &
    (tsi_df.shift(1) >= tsi_signal_df.shift(1)) & (rsi_df > 70)).shift(1).fillna(False).astype(bool)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()


Start                              2015-05-05 00:00:00
End                                2026-05-04 00:00:00
Period                              2766 days 00:00:00
Start Value                                   100000.0
End Value                                327403.835296
Total Return [%]                            227.403835
Benchmark Return [%]                        524.559266
Max Gross Exposure [%]                        92.65141
Total Fees Paid                            1381.934421
Max Drawdown [%]                             28.322899
Max Drawdown Duration                209 days 00:00:00
Total Trades                                       152
Total Closed Trades                                132
Total Open Trades                                   20
Open Trade PnL                           165902.409475
Win Rate [%]                                 15.151515
Best Trade [%]                              926.313699
Worst Trade [%]                             -17.900807
Avg Winnin

### RSI paired with TSI (or exit conditions)

In [6]:
entries = (
    (tsi_df > tsi_signal_df) &
    (tsi_df.shift(1) <= tsi_signal_df.shift(1)) & (rsi_df < 30)).shift(1).fillna(False).astype(bool)
exits = (
    (tsi_df < tsi_signal_df) &
    (tsi_df.shift(1) >= tsi_signal_df.shift(1)) | (rsi_df > 70)).shift(1).fillna(False).astype(bool)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()


Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               138674.972985
Total Return [%]                            38.674973
Benchmark Return [%]                       524.559266
Max Gross Exposure [%]                      82.729974
Total Fees Paid                           3034.687929
Max Drawdown [%]                            10.670453
Max Drawdown Duration               426 days 00:00:00
Total Trades                                      162
Total Closed Trades                               161
Total Open Trades                                   1
Open Trade PnL                             1878.22717
Win Rate [%]                                45.341615
Best Trade [%]                              85.306769
Worst Trade [%]                            -16.442089
Avg Winning Trade [%]       

### The two conditions have tradeoffs in terms of win rate, returns, sharpe ratio and max drawdown percentage